# Modèle de fréquence — probabilité de sinistre

La cible vaut 1 si le contrat a connu un sinistre. Avec environ 5,8 % de positifs, une accuracy élevée ne démontre pas une bonne tarification. Nous évaluons des probabilités avec log-loss, score de Brier, AUC et calibration.

Les résultats ci-dessous proviennent de `python run.py search`. Le code réutilisable est dans `src/pricing/`.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'run.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pricing.experiment import load_data
train, test, development, historique = load_data()
R = ROOT / 'reports'
selection = json.loads((R / 'selection.json').read_text(encoding='utf-8'))
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})


In [2]:
pd.DataFrame({'effectif': train.nombre_sinistres.value_counts(), 'proportion': train.nombre_sinistres.value_counts(normalize=True)})

,effectif,proportion
nombre_sinistres,,
0,47083,0.94166
1,2917,0.05834


## Préparation et absence de fuite

Les cibles, index et identifiants ne sont pas des prédicteurs. Le sexe est exclu. Poids/cylindrée impossibles deviennent manquants. Les zéros du second conducteur sont structurels. Les variables dérivées décrivent département, logarithmes, puissance/poids et interactions contrat/usage.

Imputation, standardisation, catégories et encodages sont appris uniquement sur le pli d’apprentissage. L’encodage cible de référence est lui-même croisé par client. Aucun client n’est présent dans deux partitions opposées.

In [3]:
assert set(development.id_client).isdisjoint(historique.id_client)
print(f'{len(development):,} contrats de développement ; {len(historique):,} de diagnostic historique.')
pd.read_csv(R / 'frequency_cv.csv').round(6)

40,013 contrats de développement ; 9,987 de diagnostic historique.


,model,log_loss,Brier,AUC
0,cat_frequency_d4,0.215244,0.054477,0.651225
1,cat_frequency_d6,0.215348,0.054469,0.649488
2,cat_frequency_d5,0.215381,0.054471,0.648949
3,cat_frequency_d3,0.215433,0.054480,0.649000
4,logistic_splines_C0.03,0.215724,0.054539,0.645659
5,logistic_splines_C0.3,0.215754,0.054541,0.646031
6,logistic_reference,0.215779,0.054547,0.645623
7,lgb_frequency_l3,0.218086,0.054813,0.626050
8,hist_previous,0.219872,0.055105,0.623136
9,lgb_frequency_l7,0.219883,0.055077,0.619246


## Modèles comparés

Régression logistique régularisée, splines et interactions ; boosting par histogrammes de référence ; CatBoost avec variables catégorielles ; LightGBM avec arbres régularisés. Aucune pondération artificielle des classes ne modifie la prévalence cible. Les poids de mélange sont choisis sur la MSE de prime, pas sur l’AUC seule.

In [4]:
[(c['name'],w) for c,w in zip(selection['frequency_configs'],selection['choice']['frequency_weights']) if w]

[('logistic_reference', 0.25), ('cat_frequency_d6', 0.75)]

In [5]:
v=pd.read_csv(R/'historical_predictions.csv')
v['decile']=pd.qcut(v.proba_pred,10,duplicates='drop')
c=v.groupby('decile',observed=True).agg(predicted=('proba_pred','mean'),observed=('claim','mean'),n=('claim','size'))
fig,ax=plt.subplots();ax.plot(c.predicted,c.observed,'o-');m=max(c.predicted.max(),c.observed.max());ax.plot([0,m],[0,m],'--',color='gray');ax.set(xlabel='Probabilité moyenne prédite',ylabel='Taux observé',title='Calibration — diagnostic historique déjà examiné');plt.show()
c

C:\Users\fourr\AppData\Local\Temp\ipykernel_63004\962270116.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.plot(c.predicted,c.observed,'o-');m=max(c.predicted.max(),c.observed.max());ax.plot([0,m],[0,m],'--',color='gray');ax.set(xlabel='Probabilité moyenne prédite',ylabel='Taux observé',title='Calibration — diagnostic historique déjà examiné');plt.show()


,predicted,observed,n
decile,,,
"(0.00244, 0.0209]",0.014228,0.013013,999
"(0.0209, 0.029]",0.024946,0.020020,999
"(0.029, 0.0382]",0.033363,0.028056,998
"(0.0382, 0.0498]",0.043882,0.033033,999
"(0.0498, 0.0595]",0.055014,0.058058,999
"(0.0595, 0.0675]",0.063624,0.065130,998
"(0.0675, 0.0751]",0.071296,0.052052,999
"(0.0751, 0.0834]",0.079154,0.078156,998
"(0.0834, 0.0966]",0.089336,0.096096,999


## Lecture

Le diagnostic historique n’est pas une nouvelle validation indépendante. La validation imbriquée de la prime dans le troisième notebook évalue toute la procédure de sélection. Des probabilités bien calibrées doivent être multipliées par les coûts conditionnels pour juger la tarification finale.